# Notebook 05 — Tokenización biológica comparada: k-mer vs no-superpuesto vs BPE

**Bootcamp Bio-LLMs · Módulo 2 · Sesión 1 de 2**
Proyecto posdoctoral CICESE — Modelos de lenguaje para venómica integrativa de *Conus*.

---

## Por qué este notebook

La metodología del proyecto establece explícitamente:

> *"Las bases de datos se estandarizarán y tokenizarán en k-meros [...] y fragmentos (H-Net) [...]. Se evaluará la eficiencia de tokenización basada en transformadores, y mecanismo de fragmentación dinámica (H-Net)."*

Este notebook hace exactamente esa evaluación de eficiencia para los **tres esquemas de tokenización clásicos** que usan los tres modelos pilares del Módulo 1:

| Esquema | Modelo que lo usa | Cómo funciona |
|---|---|---|
| **k-mer superpuesto** | DNABERT (Ji 2021) | Ventana deslizante de k bases, stride 1 |
| **k-mer NO superpuesto** | Nucleotide Transformer (Dalla-Torre 2023) | Bloques de k bases, stride k |
| **BPE** (Byte Pair Encoding) | DNABERT-2 (Zhou 2024) | Vocabulario de subcadenas de longitud variable aprendido por frecuencia |

(La fragmentación dinámica H-Net se trata en el Notebook 06.)

## Objetivos

1. Implementar **desde cero** los tres tokenizadores sobre el mismo corpus de CDS de conotoxina.
2. Entender el problema de **fuga de información (information leakage)** del k-mer superpuesto que motivó a DNABERT-2 a abandonarlo.
3. Implementar BPE desde cero con conteo incremental de pares (algoritmo eficiente).
4. Comparar empíricamente: **tasa de compresión**, distribución de longitudes de tokens, tamaño de vocabulario efectivo, y **out-of-vocabulary** en secuencias holdout.
5. Visualizar cómo cada esquema fragmenta el mismo precursor conotoxínico.

## Pre-requisitos

* Módulo 1 completado (especialmente el `CodonTokenizer` del Notebook 03).
* PyTorch no es estrictamente necesario aquí; usamos numpy, matplotlib y `collections`.

## 0. Imports y corpus

In [ ]:
import random
import math
from collections import Counter, defaultdict
from itertools import product
from typing import List, Dict, Tuple

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Imports OK")

### Generar el corpus de CDS sintéticas

Reusamos la idea del Notebook 03: precursores conotoxínicos con estructura de tres dominios (señal hidrofóbico, propéptido cargado, péptido maduro rico en cisteínas). Para datos reales, reemplaza por NCBI BioProject **PRJNA437715** / **PRJNA526781**.

In [ ]:
def synthesize_precursor():
    """Un CDS de precursor conotoxinico con estructura biologica plausible."""
    hydrophobic = ["CTG", "CTT", "GTG", "GTT", "ATC", "GCT", "GCC", "TTT", "TTC"]
    charged = ["AAA", "AAG", "AGA", "CGT", "GAT", "GAC", "GAA", "GAG", "CAT"]
    mature_rich = (["TGT", "TGC"] * 3 +  # cisteinas sobrerrepresentadas
                   ["CCA", "CCT", "GGA", "GGC", "TCA", "AGC", "ACA", "TAT", "CGT"])

    def block(pool, n):
        return "".join(random.choice(pool) for _ in range(n))

    cds = (
        "ATG"
        + block(hydrophobic, random.randint(18, 22))   # peptido senal
        + block(charged, random.randint(20, 26))        # propeptido
        + random.choice(["AAACGT", "AAAAAA", "CGTAAA"]) # sitio de corte KR/KK/RK
        + block(mature_rich, random.randint(12, 30))    # peptido maduro
        + random.choice(["TAA", "TAG", "TGA"])          # codon de paro
    )
    return cds


# Corpus completo para evaluacion, y subcorpus para entrenar el BPE.
# NOTA: BPE aprende sus fusiones a partir de un subcorpus de 800 secuencias.
# Esto es metodologicamente valido: las fusiones de alta frecuencia se
# estabilizan con relativamente pocos datos, y mantiene el notebook agil
# (BPE puro en Python es O(merges x corpus); con 800 seqs corre en segundos).
N_EVAL, N_HOLDOUT, N_BPE_TRAIN = 2000, 500, 800

eval_corpus = [synthesize_precursor() for _ in range(N_EVAL)]
holdout_corpus = [synthesize_precursor() for _ in range(N_HOLDOUT)]
bpe_train_corpus = eval_corpus[:N_BPE_TRAIN]

total_nt = sum(len(s) for s in eval_corpus)
print(f"Corpus de evaluacion: {N_EVAL} secuencias, {total_nt:,} nucleotidos")
print(f"Corpus holdout: {N_HOLDOUT} secuencias (para medir OOV)")
print(f"Subcorpus para entrenar BPE: {N_BPE_TRAIN} secuencias")
print(f"Longitud media: {np.mean([len(s) for s in eval_corpus]):.0f} nt")
print(f"\nEjemplo:\n  {eval_corpus[0]}")

## 1. k-mer superpuesto (estilo DNABERT)

### 1.1 Definición

DNABERT tokeniza con una **ventana deslizante** de tamaño k y stride 1. Para `ATGGCT` con k=3:

```
ATG  TGG  GGC  GCT
```

Nota que cada base (excepto las de los extremos) aparece en **k tokens distintos**. Esto es el origen de la *fuga de información* que veremos en la sección 1.3.

### 1.2 Vocabulario

El vocabulario son las $4^k$ permutaciones posibles más tokens especiales. Para k=6: $4^6 = 4096$ tokens + especiales.

In [ ]:
class OverlappingKmerTokenizer:
    """Tokenizador k-mer superpuesto, estilo DNABERT."""

    SPECIAL = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]

    def __init__(self, k=6):
        self.k = k
        kmers = ["".join(p) for p in product("ACGT", repeat=k)]
        self.vocab = self.SPECIAL + kmers
        self.t2i = {t: i for i, t in enumerate(self.vocab)}
        self.i2t = {i: t for t, i in self.t2i.items()}
        self.unk_id = self.t2i["[UNK]"]

    @property
    def vocab_size(self):
        return len(self.vocab)

    def tokenize(self, seq: str) -> List[str]:
        """Ventana deslizante de tamano k, stride 1."""
        seq = seq.upper()
        return [seq[i:i+self.k] for i in range(len(seq) - self.k + 1)]

    def encode(self, seq: str) -> List[int]:
        return [self.t2i.get(tok, self.unk_id) for tok in self.tokenize(seq)]


# Demo
kmer_tok = OverlappingKmerTokenizer(k=6)
example = "ATGGCTTGTTGC"
print(f"Secuencia: {example}  (longitud {len(example)} nt)")
print(f"Tokens k=6 superpuestos: {kmer_tok.tokenize(example)}")
print(f"Numero de tokens: {len(kmer_tok.tokenize(example))}  (= len - k + 1 = {len(example)-6+1})")
print(f"Tamano de vocabulario: {kmer_tok.vocab_size:,}")

### 1.3 El problema de la fuga de información (information leakage)

Este es el argumento central de DNABERT-2 (Zhou et al. 2024) contra el k-mer superpuesto.

> *"k-mer tokenization resulted in information leakage and overall poor computational efficiency during pre-training."* — DNABERT-2

**¿Por qué hay fuga?** En MLM enmascaras un token y pides al modelo que lo prediga. Pero con k-mers superpuestos, los tokens vecinos **contienen literalmente las bases del token enmascarado**.

Veamos: si enmascaras `GGC` en `ATG TGG GGC GCT`, los tokens vecinos `TGG` y `GCT` ya revelan que las posiciones comparten `GG` y `GC`. El modelo puede "hacer trampa" reconstruyendo el token enmascarado a partir del solapamiento, sin aprender semántica biológica real.

In [ ]:
def demonstrate_leakage(seq, k=3):
    """Muestra como los tokens vecinos revelan el contenido de un token enmascarado."""
    tok = OverlappingKmerTokenizer(k=k)
    tokens = tok.tokenize(seq)
    print(f"Secuencia: {seq}")
    print(f"Tokens (k={k}): {tokens}\n")

    mask_idx = len(tokens) // 2
    masked_token = tokens[mask_idx]
    print(f"Enmascaramos el token en posicion {mask_idx}: '{masked_token}' -> [MASK]\n")

    if mask_idx > 0:
        left = tokens[mask_idx - 1]
        shared_left = left[1:]
        print(f"Vecino IZQUIERDO '{left}' -> sus ultimas {k-1} bases son '{shared_left}'")
        print(f"  => revela que el token enmascarado EMPIEZA con '{shared_left}'")
    if mask_idx < len(tokens) - 1:
        right = tokens[mask_idx + 1]
        shared_right = right[:-1]
        print(f"Vecino DERECHO   '{right}' -> sus primeras {k-1} bases son '{shared_right}'")
        print(f"  => revela que el token enmascarado TERMINA con '{shared_right}'")

    if 0 < mask_idx < len(tokens) - 1:
        reconstructed = tokens[mask_idx-1][1:] + tokens[mask_idx+1][-1]
        print(f"\nReconstruccion trivial desde vecinos: '{reconstructed}'")
        print(f"Token real enmascarado            : '{masked_token}'")
        print(f"Coincide? {'SI - FUGA TOTAL' if reconstructed == masked_token else 'parcial'}")


demonstrate_leakage("ATGGCTTGT", k=3)

**Conclusión de la sección 1.** El k-mer superpuesto con MLM permite reconstruir tokens enmascarados desde los vecinos sin aprender biología. Por eso DNABERT original enmascaraba **spans contiguos de k tokens** (no tokens individuales) como mitigación parcial — pero esto no resuelve la ineficiencia computacional. NT y DNABERT-2 abandonaron el solapamiento por completo.

## 2. k-mer NO superpuesto (estilo Nucleotide Transformer)

### 2.1 Definición

NT usa bloques de k bases con **stride k** (sin solapamiento). Para `ATGGCTACG` con k=3:

```
ATG  GCT  ACG
```

Cada base aparece en **exactamente un token**. Esto elimina la fuga, y reduce la longitud tokenizada por un factor de k (importante porque la atención es O(n²)).

### 2.2 El detalle del "remainder"

Si la longitud no es múltiplo de k, sobran bases al final. NT las maneja con tokens standalone para A, T, C, G individuales (vocabulario = $4^k$ + 4 standalone + especiales).

In [ ]:
class NonOverlappingKmerTokenizer:
    """Tokenizador k-mer NO superpuesto, estilo Nucleotide Transformer."""

    SPECIAL = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]

    def __init__(self, k=6):
        self.k = k
        kmers = ["".join(p) for p in product("ACGT", repeat=k)]
        standalone = list("ACGT")  # para el remainder, como NT
        self.vocab = self.SPECIAL + kmers + standalone
        self.t2i = {t: i for i, t in enumerate(self.vocab)}
        self.i2t = {i: t for t, i in self.t2i.items()}
        self.unk_id = self.t2i["[UNK]"]

    @property
    def vocab_size(self):
        return len(self.vocab)

    def tokenize(self, seq: str) -> List[str]:
        """Bloques de k bases, stride k. El remainder se tokeniza base a base."""
        seq = seq.upper()
        tokens = []
        n_full = len(seq) // self.k
        for i in range(n_full):
            tokens.append(seq[i*self.k:(i+1)*self.k])
        for base in seq[n_full*self.k:]:  # remainder
            tokens.append(base)
        return tokens

    def encode(self, seq: str) -> List[int]:
        return [self.t2i.get(tok, self.unk_id) for tok in self.tokenize(seq)]


# Demo
nt_tok = NonOverlappingKmerTokenizer(k=6)
example = "ATGGCTTGTTGCAA"  # 14 nt, no multiplo de 6
print(f"Secuencia: {example}  (longitud {len(example)} nt)")
print(f"Tokens k=6 no-superpuestos: {nt_tok.tokenize(example)}")
print(f"  -> {len(example)//6} bloques completos + {len(example)%6} bases de remainder")
print(f"Numero de tokens: {len(nt_tok.tokenize(example))}")
print(f"Tamano de vocabulario: {nt_tok.vocab_size:,}")
print(f"\nCompara: el mismo input con k-mer SUPERPUESTO daria {len(example)-6+1} tokens")

## 3. Byte Pair Encoding (estilo DNABERT-2)

### 3.1 El algoritmo

BPE (Sennrich et al. 2016, adaptado por DNABERT-2) construye un vocabulario de subcadenas de **longitud variable** por fusión iterativa:

1. Inicializa el vocabulario con los caracteres únicos (A, C, G, T).
2. Cuenta todos los pares de tokens adyacentes en el corpus.
3. Fusiona el par más frecuente en un nuevo token.
4. Repite hasta alcanzar el tamaño de vocabulario deseado (o hasta que no queden pares con frecuencia ≥ 2).

> *"BPE [...] iteratively merging the most frequent co-occurring genome segment in the corpus [...] reduces the sequence length by approximately 5 times, substantially improving computational efficiency."* — DNABERT-2

### 3.2 Ventaja clave para MLM

Como BPE produce tokens de longitud variable, cuando enmascaras un token el modelo **no sabe cuántos nucleótidos** debe predecir. Esto transforma el MLM en un objetivo tipo T5 ("replace spans") que DNABERT-2 argumenta es más efectivo. Cero fuga de información.

### 3.3 Nota de implementación

La versión ingenua recuenta todos los pares en cada iteración, lo que es O(merges × corpus). Implementamos una versión con **conteo incremental**: tras cada fusión, sólo actualizamos los conteos de los pares afectados. Esto es lo que hace BPE de producción (HuggingFace `tokenizers`, SentencePiece).

In [ ]:
class FastBPE:
    """
    Byte Pair Encoding desde cero con conteo incremental de pares.
    Implementacion pedagogica pero eficiente.
    """

    SPECIAL = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]

    def __init__(self, vocab_size=512):
        self.target_vocab_size = vocab_size
        self.merges = []   # lista ordenada de pares fusionados
        self.vocab = None
        self.t2i = None
        self.i2t = None

    def train(self, corpus: List[str], verbose=True):
        """Aprende las fusiones BPE con conteo incremental."""
        # Cada secuencia como lista de caracteres
        words = [list(seq.upper()) for seq in corpus]
        base_chars = sorted(set(c for seq in corpus for c in seq.upper()))
        current_vocab = set(base_chars)
        n_merges = self.target_vocab_size - len(self.SPECIAL) - len(base_chars)

        if verbose:
            print(f"Caracteres base: {base_chars}")
            print(f"Fusiones objetivo: {n_merges}")

        # Conteo inicial de pares + indice inverso (que palabras contienen cada par)
        pair_counts = Counter()
        pair_to_words = defaultdict(set)
        for wi, toks in enumerate(words):
            for i in range(len(toks) - 1):
                p = (toks[i], toks[i+1])
                pair_counts[p] += 1
                pair_to_words[p].add(wi)

        for step in range(n_merges):
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=pair_counts.get)
            best_count = pair_counts[best_pair]
            if best_count < 2:  # no fusionar pares unicos
                if verbose:
                    print(f"  (detenido en merge {step}: no quedan pares frecuentes)")
                break

            self.merges.append(best_pair)
            merged = best_pair[0] + best_pair[1]
            current_vocab.add(merged)

            # Actualizar SOLO las palabras afectadas
            for wi in list(pair_to_words[best_pair]):
                toks = words[wi]
                # Retirar conteos viejos de esta palabra
                for i in range(len(toks) - 1):
                    p = (toks[i], toks[i+1])
                    pair_counts[p] -= 1
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                        pair_to_words.pop(p, None)
                    else:
                        pair_to_words[p].discard(wi)
                # Aplicar la fusion
                new_toks = []
                i = 0
                while i < len(toks):
                    if (i < len(toks) - 1 and toks[i] == best_pair[0]
                            and toks[i+1] == best_pair[1]):
                        new_toks.append(merged)
                        i += 2
                    else:
                        new_toks.append(toks[i])
                        i += 1
                words[wi] = new_toks
                # Anadir conteos nuevos
                for i in range(len(new_toks) - 1):
                    p = (new_toks[i], new_toks[i+1])
                    pair_counts[p] += 1
                    pair_to_words[p].add(wi)

            if verbose and (step < 5 or step % 200 == 0):
                print(f"  merge {step+1:>4}: {best_pair[0]}+{best_pair[1]} "
                      f"-> '{merged}' (freq {best_count})")

        self.vocab = self.SPECIAL + sorted(current_vocab)
        self.t2i = {t: i for i, t in enumerate(self.vocab)}
        self.i2t = {i: t for t, i in self.t2i.items()}
        self.unk_id = self.t2i["[UNK]"]
        if verbose:
            print(f"\nVocabulario final: {len(self.vocab)} tokens "
                  f"({len(self.merges)} fusiones aplicadas)")

    @property
    def vocab_size(self):
        return len(self.vocab) if self.vocab else 0

    def tokenize(self, seq: str) -> List[str]:
        """Aplica las fusiones aprendidas en orden a una secuencia nueva."""
        tokens = list(seq.upper())
        for pair in self.merges:
            merged = pair[0] + pair[1]
            new_tokens = []
            i = 0
            while i < len(tokens):
                if (i < len(tokens) - 1 and tokens[i] == pair[0]
                        and tokens[i+1] == pair[1]):
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return tokens

    def encode(self, seq: str) -> List[int]:
        return [self.t2i.get(tok, self.unk_id) for tok in self.tokenize(seq)]


# Entrenar BPE con vocabulario 512
print("Entrenando BPE (vocab=512) sobre subcorpus de 800 secuencias...")
bpe = FastBPE(vocab_size=512)
bpe.train(bpe_train_corpus, verbose=True)

In [ ]:
# Ver como BPE tokeniza un ejemplo
example = eval_corpus[0]
bpe_tokens = bpe.tokenize(example)
print(f"Secuencia ({len(example)} nt):\n  {example}\n")
print(f"BPE tokens ({len(bpe_tokens)} tokens):")
print(f"  {bpe_tokens}\n")

# Distribucion de longitudes de los tokens aprendidos
token_lengths = [len(t) for t in bpe.vocab if t not in FastBPE.SPECIAL]
print(f"Longitud de los tokens en el vocabulario BPE:")
print(f"  min={min(token_lengths)}, max={max(token_lengths)}, media={np.mean(token_lengths):.2f}")

# Los tokens mas largos capturan motivos recurrentes
long_tokens = sorted([t for t in bpe.vocab if t not in FastBPE.SPECIAL],
                     key=len, reverse=True)[:15]
print(f"\nTokens BPE mas largos (motivos recurrentes capturados):")
for t in long_tokens:
    print(f"  '{t}' ({len(t)} nt)")

## 4. Comparación empírica de los tres esquemas

Ahora medimos sobre el corpus holdout las cuatro métricas que importan para elegir tokenizador:

1. **Tasa de compresión**: nucleótidos / tokens. Mayor = secuencias más cortas = atención más barata.
2. **Longitud tokenizada media**: nº de tokens por secuencia.
3. **Tamaño de vocabulario**: afecta el tamaño de la matriz de embedding.
4. **Cobertura (1 - OOV)**: fracción de tokens que NO son [UNK] en holdout.

In [ ]:
def evaluate_tokenizer(tokenizer, corpus, name):
    """Calcula metricas de eficiencia sobre un corpus."""
    total_nt = total_tokens = total_unk = 0
    token_counts_per_seq = []
    unk_id = getattr(tokenizer, "unk_id", None)

    for seq in corpus:
        ids = tokenizer.encode(seq)
        total_nt += len(seq)
        total_tokens += len(ids)
        token_counts_per_seq.append(len(ids))
        if unk_id is not None:
            total_unk += sum(1 for i in ids if i == unk_id)

    return {
        "name": name,
        "vocab_size": tokenizer.vocab_size,
        "compression": total_nt / total_tokens,
        "mean_tokens_per_seq": np.mean(token_counts_per_seq),
        "oov_rate": total_unk / total_tokens if total_tokens else 0,
        "token_counts": token_counts_per_seq,
    }


# Entrenar un BPE adicional con vocab grande para contraste
print("Entrenando BPE adicional con vocab=4096 para comparar...")
bpe_large = FastBPE(vocab_size=4096)
bpe_large.train(bpe_train_corpus, verbose=False)
print(f"  (vocab efectivo: {bpe_large.vocab_size} -- puede ser < 4096 si se agotan pares frecuentes)\n")

tokenizers = {
    "k-mer superpuesto (k=6) [DNABERT]": OverlappingKmerTokenizer(k=6),
    "k-mer NO-superpuesto (k=6) [NT]": NonOverlappingKmerTokenizer(k=6),
    "BPE vocab=512 [DNABERT-2]": bpe,
    "BPE vocab=4096 [DNABERT-2]": bpe_large,
}

results = [evaluate_tokenizer(tok, holdout_corpus, name)
           for name, tok in tokenizers.items()]

print("=" * 95)
print(f"{'Esquema':<38}{'Vocab':<10}{'Compresion':<14}{'Tokens/seq':<14}{'OOV %'}")
print("=" * 95)
for r in results:
    print(f"{r['name']:<38}{r['vocab_size']:<10,}{r['compression']:<14.2f}"
          f"{r['mean_tokens_per_seq']:<14.1f}{r['oov_rate']*100:.2f}%")
print("=" * 95)
print("\nCompresion mas alta = secuencias mas cortas = atencion O(n^2) mas barata.")

In [ ]:
# Visualizacion comparativa
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

names_short = ["k-mer\nsuperpuesto", "k-mer\nno-superp.", "BPE\n512", "BPE\n4096"]
colors = ["#d62728", "#ff7f0e", "#1f77b4", "#2ca02c"]

# (a) Compresion
comps = [r["compression"] for r in results]
axes[0].bar(names_short, comps, color=colors)
axes[0].set_ylabel("Nucleotidos / token")
axes[0].set_title("(a) Tasa de compresion\n(mayor = mejor para eficiencia)")
axes[0].axhline(1.0, ls="--", color="gray", alpha=0.5, label="nivel nucleotido")
for i, v in enumerate(comps):
    axes[0].text(i, v + 0.05, f"{v:.2f}", ha="center", fontsize=10)
axes[0].legend()

# (b) Tokens por secuencia (distribucion)
for r, c in zip(results, colors):
    axes[1].hist(r["token_counts"], bins=30, alpha=0.5,
                 label=r["name"].split("[")[0].strip(), color=c)
axes[1].set_xlabel("Tokens por secuencia")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("(b) Distribucion de longitud tokenizada")
axes[1].legend(fontsize=7)

# (c) Vocab size vs compresion (trade-off)
for r, c in zip(results, colors):
    axes[2].scatter(r["vocab_size"], r["compression"], s=150, color=c)
    axes[2].annotate(r["name"].split("[")[0].strip()[:12],
                     (r["vocab_size"], r["compression"]),
                     fontsize=7, xytext=(5, 5), textcoords="offset points")
axes[2].set_xlabel("Tamano de vocabulario")
axes[2].set_ylabel("Compresion")
axes[2].set_title("(c) Trade-off vocab vs compresion")
axes[2].set_xscale("log", base=2)

plt.tight_layout()
plt.show()

## 5. Visualización: ¿cómo fragmenta cada esquema el mismo precursor?

Tomamos un precursor y mostramos su segmentación lado a lado. Las **cisteínas (TGT/TGC)** del péptido maduro son la firma funcional de la conotoxina.

In [ ]:
def visualize_segmentation(seq, tokenizers_dict, max_display_nt=48):
    """Imprime la segmentacion de una secuencia bajo cada tokenizador."""
    seq = seq[:max_display_nt]
    print(f"Secuencia (primeros {len(seq)} nt):")
    print(f"  {seq}\n")
    for name, tok in tokenizers_dict.items():
        tokens = tok.tokenize(seq)
        segmented = "|".join(tokens)
        print(f"{name}:")
        print(f"  {segmented}")
        print(f"  ({len(tokens)} tokens)\n")


viz_tokenizers = {
    "k-mer superpuesto (k=6)": OverlappingKmerTokenizer(k=6),
    "k-mer no-superpuesto (k=6)": NonOverlappingKmerTokenizer(k=6),
    "BPE vocab=512": bpe,
    "BPE vocab=4096": bpe_large,
}

sample = eval_corpus[3]
visualize_segmentation(sample, viz_tokenizers, max_display_nt=48)

## 6. Interpretación de resultados

### Qué deberías observar

| Esquema | Compresión | OOV | Veredicto |
|---|---|---|---|
| **k-mer superpuesto** | ~1.0 (no comprime; nº tokens ≈ nº bases) | 0% | Fuga de información, atención cara. **Evitar para nuevos proyectos** |
| **k-mer no-superpuesto** | ~k (≈6 con k=6) | 0% | Sin fuga, compresión fija. El default robusto de NT |
| **BPE 512** | Variable (~4-5) | Bajo | Compresión adaptativa, captura motivos. Vocab pequeño |
| **BPE 4096** | Más alta (~6) | Bajo | Mejor compresión, vocab mayor. La elección de DNABERT-2 |

### Una observación importante sobre BPE 4096

Es posible que el vocabulario efectivo del BPE 4096 sea **menor que 4096**. Esto pasa cuando se agotan los pares con frecuencia ≥ 2 antes de alcanzar el objetivo — el corpus sintético no tiene suficiente diversidad de motivos largos. En un corpus real multi-especie de *Conidae*, el vocabulario se llenaría más. Esta es una lección genuina: **el tamaño de vocabulario alcanzable depende de la riqueza del corpus**.

### Decisión para tu proyecto

La metodología del proyecto dice que se *"evaluará la eficiencia de tokenización basada en transformadores"*. Con base en este notebook:

* Para el **gLM** (genómico, secuencias largas): BPE o k-mer no-superpuesto. BPE gana en compresión, crítico cuando proceses exomas largos donde la atención O(n²) duele.
* Para el **pLM** (proteómico, conotoxinas cortas <50 aa): aquí la compresión importa menos. La tokenización a nivel aminoácido (como ESM-2) suele ser suficiente, y preserva la resolución por residuo necesaria para predecir cisteínas individuales.

**Advertencia sobre BPE en proteínas.** Boshar et al. (2024) muestran que para tareas que requieren resolución por residuo (como contactos disulfuro), la tokenización gruesa puede perjudicar. El 3-mer de NT-v2 supera al 6-mer en tareas proteómicas finas. Para conotoxinas, donde cada cisteína cuenta, considera resolución fina.

## 7. Ejercicios

### 7.1 — Barrido de vocab BPE (replicar Figura 3a de DNABERT-2)
DNABERT-2 construyó 8 vocabularios de $2^8$ a $2^{15}$. Entrena BPE con `vocab_size ∈ {256, 1024, 4096, 8192}` y grafica compresión vs vocab. Confirma que vocabularios mayores comprimen más (hasta saturar por falta de motivos en el corpus sintético).

### 7.2 — k variable en k-mer no-superpuesto
Mide compresión y OOV para `k ∈ {3, 4, 5, 6}` con el tokenizador NT. ¿Cuándo empieza a subir el OOV en holdout?

### 7.3 — Demostrar la fuga cuantitativamente
Entrena un clasificador trivial que, dados los dos tokens vecinos, prediga el token enmascarado, para k-mer superpuesto vs no-superpuesto. La accuracy debe ser ~100% para superpuesto y ~baseline para no-superpuesto.

### 7.4 — BPE sobre aminoácidos
Adapta `FastBPE` para operar sobre secuencias de aminoácidos (alfabeto de 20) de ConoServer. ¿Qué motivos captura? ¿Aparecen patrones cisteínicos como tokens largos?

### 7.5 — Tokenización de codones vs BPE
Compara el `CodonTokenizer` del Notebook 03 (3-mer no-superpuesto, alineado a marco de lectura) con BPE. El codón respeta la biología (marco de lectura); BPE no. ¿Cuál comprime más? ¿Cuál preserva mejor la estructura del CDS?

## 8. Conclusión y siguiente notebook

✓ Implementaste **desde cero** los tres tokenizadores clásicos de los modelos pilares.
✓ Demostraste la **fuga de información** del k-mer superpuesto que motivó su abandono.
✓ Implementaste **BPE** completo con conteo incremental de pares (algoritmo de producción).
✓ Comparaste empíricamente compresión, longitud tokenizada, vocab y OOV.

### Notebook 06 — Fragmentación dinámica (H-Net) y el impacto downstream

En el siguiente notebook:
* Implementaremos la idea de **fragmentación dinámica de H-Net** (chunking aprendido sin vocabulario fijo), el tercer mecanismo que tu metodología menciona explícitamente.
* Conectaremos cada tokenizador a un MLM mínimo y mediremos el **impacto downstream**: ¿la elección de tokenizador cambia la perplejidad y la calidad de los embeddings?

### Bibliografía de este notebook

* **Ji et al. (2021)** DNABERT — Bioinformatics 37(15). (k-mer superpuesto)
* **Dalla-Torre et al. (2025)** Nucleotide Transformer — Nature Methods 22:287–297. (k-mer no-superpuesto 6-mer)
* **Zhou et al. (2024)** DNABERT-2 — ICLR. (BPE)
* **Sennrich et al. (2016)** Neural Machine Translation of Rare Words with Subword Units — ACL. (BPE original)
* **Boshar et al. (2024)** Are genomic language models all you need? — Bioinformatics 40(9):btae529. (3-mer vs 6-mer en proteínas)

In [ ]:
print("=" * 60)
print("RESUMEN MODULO 2 - NOTEBOOK 05")
print("=" * 60)
for r in results:
    print(f"  {r['name'][:36]:<38} compresion={r['compression']:.2f}  OOV={r['oov_rate']*100:.1f}%")
print("\nListo para Notebook 06 (H-Net + impacto downstream).")